In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
import re
import random
import numpy as np
import polars as pl
import torch
from tqdm import tqdm

TRAIN_PATH = "/content/train.parquet"
QUERIES_PATH = "/content/benchmark_queries.parquet"
ADS_PATH = "/content/benchmark_items.parquet"
CACHE_DIR = "/content/drive/MyDrive/avito_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [26]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
print(model.device)

def encode(texts, prefix):
    return model.encode(
        [prefix + t for t in texts],
        batch_size=256,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=len(texts) > 1000,
    ).astype(np.float32)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

cuda:0


In [27]:
NEXT_KEYS = [
    "Тип стоимости", "Тип услуги", "Вид услуги", "Начальная цена", "Время для связи",
    "Время работы", "График работы", "Рабочие дни", "Марка", "Название услуги",
    "Своя услуга", "Услуга", "Опыт работы", "Стоимость", "Продолжительность",
    "Как вы работаете", "Куда выезжаете", "Работаете с", "Работа по договору",
    "Готов закупить", "Чем вы", "Выполняю заказы", "Гарантия", "Бригада",
    "Год окончания", "Учебное учреждение", "Берёте ли", "Онлайн-показ",
    "Проживание на объекте", "Предмет или специальность", "Предоплата",
    "Есть портфолио", "Специализация",
]

RATING_RE = re.compile(r"Рейтинг пользователя\s+\d+(?:[.,]\d+)?\s+звезд\w*\s+и\s+выше(?:\s*\[поиск\])?")
ADDR_RE = re.compile(
    r"Место оказания услуг.*?(?=\s(?:" + "|".join(map(re.escape, NEXT_KEYS)) + r")(?:\s|,|$)|$)"
)

def clean_spaces(s):
    return re.sub(r"\s+", " ", s).strip()

def cut_rating(s):
    return clean_spaces(RATING_RE.sub(" ", s)) if isinstance(s, str) else ""

def cut_address(s):
    return clean_spaces(ADDR_RE.sub(" ", s)) if isinstance(s, str) else ""

In [28]:
ads = pl.read_parquet(ADS_PATH, columns=[
    "item_id", "item_title_raw", "item_infm_params_text",
    "item_location_id", "item_category_id", "item_rating",
])

ITEM_IDS = ads["item_id"].to_numpy()
ITEM_LOC = ads["item_location_id"].to_numpy()
ITEM_CAT = ads["item_category_id"].to_numpy()
ITEM_RATING = ads["item_rating"].cast(pl.Float64).fill_null(np.nan).to_numpy()

titles = ads["item_title_raw"].fill_null("").to_list()
params = [cut_address(s) for s in ads["item_infm_params_text"].fill_null("").to_list()]

def cached_encode(name, texts, prefix):
    path = f"{CACHE_DIR}/{name}.npy"
    if os.path.exists(path):
        return np.load(path)
    E = encode(texts, prefix)
    np.save(path, E)
    return E

tag = MODEL_NAME.split("/")[-1]
E_TITLE = cached_encode(f"{tag}_title", titles, "passage: ")
E_PARAMS = cached_encode(f"{tag}_params", params, "query: ")
print(E_TITLE.shape, E_PARAMS.shape)

(189212, 384) (189212, 384)


In [29]:
RATING_THR_RE = re.compile(r"Рейтинг[^0-9]*(\d(?:[.,]\d)?)")
POOL_MASK = np.ones(len(ITEM_IDS), dtype=bool)

def hard_filter(search_category, filters_text):
    mask = POOL_MASK & (ITEM_CAT == search_category)
    m = RATING_THR_RE.search(filters_text or "")
    if m:
        thr = float(m.group(1).replace(",", "."))
        mask &= np.isnan(ITEM_RATING) | (ITEM_RATING >= thr)
    return np.flatnonzero(mask)

def split_by_location(search_location_id, allowed):
    local = ITEM_LOC[allowed] == search_location_id
    return allowed[local], allowed[~local]

In [30]:
train_h = pl.read_parquet(TRAIN_PATH, columns=[
    "search_query", "search_infm_params_text", "search_location_id", "item_location_id",
])

TR_SLOC = train_h["search_location_id"].to_numpy()
TR_ILOC = train_h["item_location_id"].to_numpy()

tr_queries = train_h["search_query"].fill_null("").to_list()
tr_filters = [cut_rating(s) for s in train_h["search_infm_params_text"].fill_null("").to_list()]

def encode_unique(name, texts, prefix):
    uniq, inv = np.unique(np.array(texts, dtype=object), return_inverse=True)
    return cached_encode(name, list(uniq), prefix), inv

E_TR_Q, TR_Q_INV = encode_unique(f"{tag}_train_query", tr_queries, "query: ")
E_TR_F, TR_F_INV = encode_unique(f"{tag}_train_filters", tr_filters, "query: ")
print(E_TR_Q.shape, E_TR_F.shape)

TR_ACTIVE = np.ones(len(TR_SLOC), dtype=bool)

def locations_from_history(search_location_id, q_title, q_filters, x_candidates):
    rows = np.flatnonzero((TR_SLOC == search_location_id) & TR_ACTIVE)
    if len(rows) == 0:
        return []
    prox_query = E_TR_Q[TR_Q_INV[rows]] @ q_title
    prox_filters = (E_TR_F[TR_F_INV[rows]] @ q_filters if q_filters is not None
                    else np.zeros(len(rows), dtype=np.float32))
    order = np.lexsort((-prox_filters, -prox_query))[:x_candidates]
    return np.unique(TR_ILOC[rows[order]]).tolist()

(74529, 384) (3519, 384)


In [31]:
def top_n(idx, prox_title, prox_filters, n, is_local):
    k = min(n, len(idx))
    if k == 0:
        return {}
    part = np.argpartition(-prox_title, k - 1)[:k] if k < len(idx) else np.arange(len(idx))
    order = part[np.lexsort((-prox_filters[part], -prox_title[part]))]
    return {ITEM_IDS[idx[i]]: (float(prox_title[i]), float(prox_filters[i]), is_local)
            for i in order}

def search(query_text, filters_text, _LOCAL, _NON_LOCAL, n_candidates,
           search_location_id=None, x_candidates=50):
    q_title = encode([query_text or ""], "query: ")[0]
    filters_text = cut_rating(filters_text)
    q_filters = encode([filters_text], "query: ")[0] if filters_text else None

    if len(_LOCAL) == 0 and search_location_id is not None:
        locs = locations_from_history(search_location_id, q_title, q_filters, x_candidates)
        allowed = np.concatenate([_LOCAL, _NON_LOCAL])
        local = np.isin(ITEM_LOC[allowed], locs)
        _LOCAL, _NON_LOCAL = allowed[local], allowed[~local]

    def proximity(idx, is_local):
        prox_title = E_TITLE[idx] @ q_title
        prox_filters = (E_PARAMS[idx] @ q_filters if q_filters is not None
                        else np.zeros(len(idx), dtype=np.float32))
        return top_n(idx, prox_title, prox_filters, n_candidates, is_local)

    _LOCAL_proximity = proximity(_LOCAL, True)
    _NON_LOCAL_proximity = proximity(_NON_LOCAL, False)

    merged = {**_LOCAL_proximity, **_NON_LOCAL_proximity}
    return dict(sorted(merged.items(), key=lambda kv: kv[1], reverse=True))

Валидация
===

In [33]:
Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]
ITEM_COLS = ["item_id", "item_title_raw", "item_infm_params_text",
             "item_location_id", "item_category_id", "item_rating"]
N_VAL = 2500
N_RANK = 10000

tr = (pl.read_parquet(TRAIN_PATH, columns=Q_COLS + ITEM_COLS)
      .with_columns(pl.col("search_query", "search_infm_params_text").fill_null(""))
      .with_row_index("row"))

groups = tr.select(Q_COLS).unique(maintain_order=True).with_row_index("qid")
tr = tr.join(groups, on=Q_COLS, how="left").sort("row")

rng = np.random.default_rng(SEED)
perm = rng.permutation(groups.height)
val_qids = perm[:N_VAL]
rank_qids = perm[N_VAL:N_VAL + N_RANK]

def split_part(qids):
    rows = tr.filter(pl.col("qid").is_in(qids))
    qs = groups.filter(pl.col("qid").is_in(qids))
    rel = {qid: set(items) for qid, items in
           rows.group_by("qid").agg(pl.col("item_id").unique()).iter_rows()}
    return rows, qs, rel

val_rows, val_queries, VAL_REL = split_part(val_qids)
rank_rows, rank_queries, RANK_REL = split_part(rank_qids)

TR_ACTIVE = ~np.isin(tr["qid"].to_numpy(), np.concatenate([val_qids, rank_qids]))
print("валидация:", val_queries.height, "| обучение ранкера:", rank_queries.height,
      "| строк в истории:", TR_ACTIVE.sum())

валидация: 2500 | обучение ранкера: 10000 | строк в истории: 480009


In [34]:
if "BASE" not in globals():
    BASE = dict(ids=ITEM_IDS, loc=ITEM_LOC, cat=ITEM_CAT, rating=ITEM_RATING,
                title=E_TITLE, params=E_PARAMS)
N_CORPUS = len(BASE["ids"])

new_items = (pl.concat([val_rows, rank_rows])
             .filter(~pl.col("item_id").is_in(BASE["ids"].tolist()))
             .unique("item_id", keep="first", maintain_order=True))

new_titles = new_items["item_title_raw"].fill_null("").to_list()
new_params = [cut_address(s) for s in new_items["item_infm_params_text"].fill_null("").to_list()]

ITEM_IDS = np.concatenate([BASE["ids"], new_items["item_id"].to_numpy()])
ITEM_LOC = np.concatenate([BASE["loc"], new_items["item_location_id"].to_numpy()])
ITEM_CAT = np.concatenate([BASE["cat"], new_items["item_category_id"].to_numpy()])
ITEM_RATING = np.concatenate([BASE["rating"],
    new_items["item_rating"].cast(pl.Float64).fill_null(np.nan).to_numpy()])
E_TITLE = np.vstack([BASE["title"], encode(new_titles, "passage: ")])
E_PARAMS = np.vstack([BASE["params"], encode(new_params, "query: ")])

POOL_MASK = np.ones(len(ITEM_IDS), dtype=bool)
print("корпус:", N_CORPUS, "| добавлено из валидации:", new_items.height, "| всего:", len(ITEM_IDS))

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

корпус: 189212 | добавлено из валидации: 15298 | всего: 204510


In [35]:
from tqdm.auto import tqdm

def evaluate(n_candidates, x_candidates, queries_df=val_queries):
    rows = []
    for q in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
        allowed = hard_filter(q["search_category"], q["search_infm_params_text"])
        _LOCAL, _NON_LOCAL = split_by_location(q["search_location_id"], allowed)
        res = search(q["search_query"], q["search_infm_params_text"], _LOCAL, _NON_LOCAL,
                     n_candidates, q["search_location_id"], x_candidates)
        rel = VAL_REL[q["qid"]]
        rows.append({"qid": q["qid"], "had_local": len(_LOCAL) > 0,
                     "recall": len(rel & res.keys()) / len(rel)})
    return pl.DataFrame(rows)

ev = evaluate(n_candidates=25, x_candidates=50)
print("Recall@50:", ev["recall"].mean())
print(ev.group_by("had_local").agg(pl.len(), pl.col("recall").mean()))

  0%|          | 0/2500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [32]:
def evaluate(n_candidates, x_candidates, queries_df=val_queries):
    rows = []
    for q in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
        allowed = hard_filter(q["search_category"], q["search_infm_params_text"])
        _LOCAL, _NON_LOCAL = split_by_location(q["search_location_id"], allowed)
        res = search(q["search_query"], q["search_infm_params_text"], _LOCAL, _NON_LOCAL,
                     n_candidates, q["search_location_id"], x_candidates)
        rel = VAL_REL[q["qid"]]
        pool = set(res)
        top50 = set(list(res)[:50])
        rows.append({"qid": q["qid"], "had_local": len(_LOCAL) > 0,
                     "recall_pool": len(rel & pool) / len(rel),
                     "recall_50": len(rel & top50) / len(rel)})
    return pl.DataFrame(rows)

ev = evaluate(n_candidates=250, x_candidates=50)
print("Recall пула:", ev["recall_pool"].mean())
print("Recall@50 по косинусу:", ev["recall_50"].mean())
print(ev.group_by("had_local").agg(pl.len(), pl.col("recall_pool").mean(), pl.col("recall_50").mean()))

  0%|          | 0/2500 [00:00<?, ?it/s]

Recall пула: 0.8914866233766233
Recall@50 по косинусу: 0.25485017316017317
shape: (2, 4)
┌───────────┬──────┬─────────────┬───────────┐
│ had_local ┆ len  ┆ recall_pool ┆ recall_50 │
│ ---       ┆ ---  ┆ ---         ┆ ---       │
│ bool      ┆ u32  ┆ f64         ┆ f64       │
╞═══════════╪══════╪═════════════╪═══════════╡
│ true      ┆ 2199 ┆ 0.904162    ┆ 0.24676   │
│ false     ┆ 301  ┆ 0.798885    ┆ 0.313953  │
└───────────┴──────┴─────────────┴───────────┘


In [37]:
def build_pool_table(queries_df, rel, n_candidates, x_candidates):
    cols = {"qid": [], "item_id": [], "prox_title": [], "prox_filters": [],
            "is_local": [], "label": []}
    for q in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
        allowed = hard_filter(q["search_category"], q["search_infm_params_text"])
        _LOCAL, _NON_LOCAL = split_by_location(q["search_location_id"], allowed)
        res = search(q["search_query"], q["search_infm_params_text"], _LOCAL, _NON_LOCAL,
                     n_candidates, q["search_location_id"], x_candidates)
        r = rel[q["qid"]]
        for item_id, (pt, pf, il) in res.items():
            cols["qid"].append(q["qid"])
            cols["item_id"].append(item_id)
            cols["prox_title"].append(pt)
            cols["prox_filters"].append(pf)
            cols["is_local"].append(il)
            cols["label"].append(int(item_id in r))
    return pl.DataFrame(cols)

N_CAND = 250
X_CAND = 50

pool_val = build_pool_table(val_queries, VAL_REL, N_CAND, X_CAND)
pool_val.write_parquet(f"{CACHE_DIR}/pool_val.parquet")

pool_rank = build_pool_table(rank_queries, RANK_REL, N_CAND, X_CAND)
pool_rank.write_parquet(f"{CACHE_DIR}/pool_rank.parquet")

print(pool_val.shape, pool_rank.shape)

  0%|          | 0/2500 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

(1167129, 6) (4655039, 6)


In [38]:
def recall_from_table(t, rel, n_local=None, n_non=None):
    if n_local is not None:
        t = (t.sort(["qid", "is_local", "prox_title", "prox_filters"],
                    descending=[False, True, True, True])
              .with_columns(pl.int_range(pl.len()).over(["qid", "is_local"]).alias("rk"))
              .filter(pl.when(pl.col("is_local")).then(pl.col("rk") < n_local)
                        .otherwise(pl.col("rk") < n_non)))
    hits = dict(t.group_by("qid").agg(pl.col("label").sum()).iter_rows())
    return np.mean([hits.get(qid, 0) / len(items) for qid, items in rel.items()])

print("валидация, recall пула:", recall_from_table(pool_val, VAL_REL))
print("валидация, 25/25:", recall_from_table(pool_val, VAL_REL, 25, 25))
print("обучение, recall пула:", recall_from_table(pool_rank, RANK_REL))
print("доля позитивов в обучении:", pool_rank["label"].mean())

валидация, recall пула: 0.8898343374272787
валидация, 25/25: 0.7229924096165272
обучение, recall пула: 0.8841164699216333
доля позитивов в обучении: 0.002525864982011966


In [39]:
np.save(f"{CACHE_DIR}/val_qids.npy", val_qids)
np.save(f"{CACHE_DIR}/rank_qids.npy", rank_qids)

rel_to_df = lambda rel: pl.DataFrame(
    [(q, i) for q, items in rel.items() for i in items], schema=["qid", "item_id"], orient="row")
rel_to_df(VAL_REL).write_parquet(f"{CACHE_DIR}/val_rel.parquet")
rel_to_df(RANK_REL).write_parquet(f"{CACHE_DIR}/rank_rel.parquet")

groups.write_parquet(f"{CACHE_DIR}/groups.parquet")

np.save(f"{CACHE_DIR}/ext_item_ids.npy", ITEM_IDS.astype(str))
np.save(f"{CACHE_DIR}/ext_title.npy", E_TITLE)
np.save(f"{CACHE_DIR}/ext_params.npy", E_PARAMS)

print(os.listdir(CACHE_DIR))

['multilingual-e5-small_title.npy', 'multilingual-e5-small_params.npy', 'multilingual-e5-small_train_query.npy', 'multilingual-e5-small_train_filters.npy', 'pool_val.parquet', 'pool_rank.parquet', 'val_qids.npy', 'rank_qids.npy', 'val_rel.parquet', 'rank_rel.parquet', 'groups.parquet', 'ext_item_ids.npy', 'ext_title.npy', 'ext_params.npy']


Признаки
===

In [16]:
Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]
ITEM_NUM = ["item_id", "item_rating", "item_rating_reviews_count", "item_price",
            "item_microcat_id", "item_latitude", "item_longitude", "item_location_id",
            "item_is_phone_hidden", "item_is_message_forbidden"]
tag = "multilingual-e5-small"

pool_val = pl.read_parquet(f"{CACHE_DIR}/pool_val.parquet")
pool_rank = pl.read_parquet(f"{CACHE_DIR}/pool_rank.parquet")
groups = pl.read_parquet(f"{CACHE_DIR}/groups.parquet")
val_qids = np.load(f"{CACHE_DIR}/val_qids.npy")
rank_qids = np.load(f"{CACHE_DIR}/rank_qids.npy")
FLOAT_COLS = ["item_rating", "item_rating_reviews_count", "item_price",
              "item_latitude", "item_longitude"]

items = (pl.concat([pl.read_parquet(ADS_PATH, columns=ITEM_NUM),
                    pl.read_parquet(TRAIN_PATH, columns=ITEM_NUM)], how="vertical_relaxed")
         .with_columns(pl.col(FLOAT_COLS).cast(pl.Float64))
         .unique("item_id", keep="first", maintain_order=True))
print(pool_val.shape, pool_rank.shape, items.shape)

(1167129, 6) (4655039, 6) (515895, 10)


In [17]:
centers = (pl.read_parquet(ADS_PATH, columns=["item_location_id", "item_latitude", "item_longitude"])
           .with_columns(pl.col("item_latitude", "item_longitude").cast(pl.Float64))
           .group_by("item_location_id")
           .agg(pl.col("item_latitude").median().alias("c_lat"),
                pl.col("item_longitude").median().alias("c_lon"))
           .rename({"item_location_id": "search_location_id"}))

qinfo = (groups.select("qid", "search_location_id")
         .join(centers, on="search_location_id", how="left")
         .with_columns(pl.col("c_lat").is_not_null().cast(pl.Int8).alias("had_local"))
         .drop("search_location_id"))
print(qinfo["had_local"].mean())

0.8753184394421984


In [18]:
hist = (pl.read_parquet(TRAIN_PATH, columns=Q_COLS + ["item_microcat_id"])
        .with_columns(pl.col("search_query", "search_infm_params_text").fill_null(""))
        .join(groups.select(Q_COLS + ["qid"]), on=Q_COLS, how="left"))

uniq_q = np.unique(np.array(hist["search_query"].to_list(), dtype=object))
E_TR_Q = np.load(f"{CACHE_DIR}/{tag}_train_query.npy")
assert len(uniq_q) == len(E_TR_Q)
t2i = {t: i for i, t in enumerate(uniq_q)}

mc_vals = np.sort(items["item_microcat_id"].drop_nulls().unique().to_numpy())
mc2i = {m: i for i, m in enumerate(mc_vals)}

active = hist.filter(~pl.col("qid").is_in(np.concatenate([val_qids, rank_qids])))
active = active.filter(pl.col("item_microcat_id").is_not_null())

from scipy.sparse import csr_matrix
C = csr_matrix((np.ones(active.height, dtype=np.float32),
                ([t2i[t] for t in active["search_query"].to_list()],
                 [mc2i[m] for m in active["item_microcat_id"].to_list()])),
               shape=(len(uniq_q), len(mc_vals)))

K_SIM = 50
target = groups.filter(pl.col("qid").is_in(np.concatenate([val_qids, rank_qids])))
tq = np.array([t2i[t] for t in target["search_query"].to_list()])
tqid = target["qid"].to_numpy()

parts = []
for s in tqdm(range(0, len(tq), 512)):
    S = E_TR_Q[tq[s:s + 512]] @ E_TR_Q.T
    top = np.argpartition(-S, K_SIM, axis=1)[:, :K_SIM]
    agg = np.vstack([np.asarray(C[t].sum(axis=0)).ravel() for t in top])
    tot = agg.sum(axis=1, keepdims=True)
    share = np.divide(agg, tot, out=np.zeros_like(agg), where=tot > 0)
    qi, mi = np.nonzero(share)
    parts.append(pl.DataFrame({"qid": tqid[s:s + 512][qi],
                               "item_microcat_id": mc_vals[mi],
                               "mc_share": share[qi, mi]}))

mc_share = pl.concat(parts)
del hist, active, S
print(mc_share.shape)

100%|██████████| 25/25 [00:17<00:00,  1.42it/s]

(207465, 3)


In [19]:
R_EARTH = 6371.0

def build_features(pool):
    lat1, lon1 = pl.col("c_lat").radians(), pl.col("c_lon").radians()
    lat2, lon2 = pl.col("item_latitude").radians(), pl.col("item_longitude").radians()
    a = ((lat2 - lat1) / 2).sin() ** 2 + lat1.cos() * lat2.cos() * ((lon2 - lon1) / 2).sin() ** 2
    return (pool
            .join(items, on="item_id", how="left")
            .join(qinfo, on="qid", how="left")
            .join(mc_share, on=["qid", "item_microcat_id"], how="left")
            .with_columns(
                pl.col("mc_share").fill_null(0.0),
                (2 * R_EARTH * a.sqrt().arcsin()).alias("dist_km"),
                pl.col("item_price").log1p().alias("log_price"),
                pl.col("item_rating_reviews_count").log1p().alias("log_reviews"),
                pl.col("is_local").cast(pl.Int8),
                pl.col("item_is_phone_hidden").cast(pl.Int8),
                pl.col("item_is_message_forbidden").cast(pl.Int8),
                pl.col("prox_title").rank("ordinal", descending=True)
                  .over(["qid", "is_local"]).alias("rank_in_stream"),
                (pl.col("prox_title").max().over("qid") - pl.col("prox_title")).alias("gap_title"),
                (pl.col("prox_filters").max().over("qid") - pl.col("prox_filters")).alias("gap_filters"),
            )
            .drop("c_lat", "c_lon", "item_latitude", "item_longitude", "item_location_id",
                  "item_price", "item_rating_reviews_count"))

feat_val = build_features(pool_val)
feat_rank = build_features(pool_rank)
feat_val.write_parquet(f"{CACHE_DIR}/feat_val.parquet")
feat_rank.write_parquet(f"{CACHE_DIR}/feat_rank.parquet")

print(feat_val.columns)
print(feat_val.null_count())

['qid', 'item_id', 'prox_title', 'prox_filters', 'is_local', 'label', 'item_rating', 'item_microcat_id', 'item_is_phone_hidden', 'item_is_message_forbidden', 'had_local', 'mc_share', 'dist_km', 'log_price', 'log_reviews', 'rank_in_stream', 'gap_title', 'gap_filters']
shape: (1, 18)
┌─────┬─────────┬────────────┬─────────────┬───┬─────────────┬────────────┬───────────┬────────────┐
│ qid ┆ item_id ┆ prox_title ┆ prox_filter ┆ … ┆ log_reviews ┆ rank_in_st ┆ gap_title ┆ gap_filter │
│ --- ┆ ---     ┆ ---        ┆ s           ┆   ┆ ---         ┆ ream       ┆ ---       ┆ s          │
│ u32 ┆ u32     ┆ u32        ┆ ---         ┆   ┆ u32         ┆ ---        ┆ u32       ┆ ---        │
│     ┆         ┆            ┆ u32         ┆   ┆             ┆ u32        ┆           ┆ u32        │
╞═════╪═════════╪════════════╪═════════════╪═══╪═════════════╪════════════╪═══════════╪════════════╡
│ 0   ┆ 0       ┆ 0          ┆ 0           ┆ … ┆ 74320       ┆ 0          ┆ 0         ┆ 0          │
└─────┴───

In [21]:
with pl.Config(tbl_rows=-1):
    print(feat_val.null_count().transpose(include_header=True, column_names=["nulls"]))

shape: (18, 2)
┌───────────────────────────┬────────┐
│ column                    ┆ nulls  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ qid                       ┆ 0      │
│ item_id                   ┆ 0      │
│ prox_title                ┆ 0      │
│ prox_filters              ┆ 0      │
│ is_local                  ┆ 0      │
│ label                     ┆ 0      │
│ item_rating               ┆ 103059 │
│ item_microcat_id          ┆ 0      │
│ item_is_phone_hidden      ┆ 0      │
│ item_is_message_forbidden ┆ 0      │
│ had_local                 ┆ 0      │
│ mc_share                  ┆ 0      │
│ dist_km                   ┆ 157372 │
│ log_price                 ┆ 0      │
│ log_reviews               ┆ 74320  │
│ rank_in_stream            ┆ 0      │
│ gap_title                 ┆ 0      │
│ gap_filters               ┆ 0      │
└───────────────────────────┴────────┘


In [22]:
import lightgbm as lgb

FEATS = [c for c in feat_val.columns if c not in ("qid", "item_id", "label")]
CAT_FEATS = ["item_microcat_id"]

def keep_with_positive(df):
    has_pos = df.group_by("qid").agg(pl.col("label").max()).filter(pl.col("label") == 1)["qid"]
    return df.filter(pl.col("qid").is_in(has_pos))

def to_lgb(df):
    df = df.sort("qid")
    X = df.select(FEATS).to_pandas()
    X[CAT_FEATS] = X[CAT_FEATS].astype("category")
    return X, df["label"].to_numpy(), df.group_by("qid", maintain_order=True).len()["len"].to_numpy()

rank_q = feat_rank["qid"].unique().sort().to_numpy()
rng = np.random.default_rng(SEED)
es_q = rng.choice(rank_q, int(len(rank_q) * 0.1), replace=False)

tr_df = keep_with_positive(feat_rank.filter(~pl.col("qid").is_in(es_q)))
es_df = keep_with_positive(feat_rank.filter(pl.col("qid").is_in(es_q)))

X_tr, y_tr, g_tr = to_lgb(tr_df)
X_es, y_es, g_es = to_lgb(es_df)
print(X_tr.shape, X_es.shape, FEATS)

/tmp/ipykernel_59494/2505621976.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  return df.filter(pl.col("qid").is_in(has_pos))


(3752873, 15) (411320, 15) ['prox_title', 'prox_filters', 'is_local', 'item_rating', 'item_microcat_id', 'item_is_phone_hidden', 'item_is_message_forbidden', 'had_local', 'mc_share', 'dist_km', 'log_price', 'log_reviews', 'rank_in_stream', 'gap_title', 'gap_filters']


In [23]:
params = dict(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[50],
    lambdarank_truncation_level=60,
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    seed=SEED,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)

d_tr = lgb.Dataset(X_tr, y_tr, group=g_tr, categorical_feature=CAT_FEATS)
d_es = lgb.Dataset(X_es, y_es, group=g_es, categorical_feature=CAT_FEATS, reference=d_tr)

ranker = lgb.train(params, d_tr, num_boost_round=3000, valid_sets=[d_es],
                   callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])
ranker.save_model(f"{CACHE_DIR}/lgb_ranker.txt")

Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@50: 0.664386
[200]	valid_0's ndcg@50: 0.665202
Early stopping, best iteration is:
[124]	valid_0's ndcg@50: 0.666615


In [24]:
val_rel = pl.read_parquet(f"{CACHE_DIR}/val_rel.parquet")
VAL_REL = {q: set(i) for q, i in val_rel.group_by("qid").agg(pl.col("item_id")).iter_rows()}

X_val = feat_val.select(FEATS).to_pandas()
X_val[CAT_FEATS] = X_val[CAT_FEATS].astype("category")
scored = feat_val.with_columns(pl.Series("score", ranker.predict(X_val, num_iteration=ranker.best_iteration)))

top50 = (scored.sort(["qid", "score"], descending=[False, True])
         .group_by("qid", maintain_order=True).head(50))
hits = dict(top50.group_by("qid").agg(pl.col("label").sum()).iter_rows())
recall = np.mean([hits.get(q, 0) / len(items) for q, items in VAL_REL.items()])

print("Recall@50 LightGBM:", recall, "| бейзлайн 25/25: 0.723 | потолок пула: 0.890")

imp = pl.DataFrame({"feature": ranker.feature_name(),
                    "gain": ranker.feature_importance("gain")}).sort("gain", descending=True)
with pl.Config(tbl_rows=-1):
    print(imp)

Recall@50 LightGBM: 0.8505358586511528 | бейзлайн 25/25: 0.723 | потолок пула: 0.890
shape: (15, 2)
┌───────────────────────────┬───────────────┐
│ feature                   ┆ gain          │
│ ---                       ┆ ---           │
│ str                       ┆ f64           │
╞═══════════════════════════╪═══════════════╡
│ rank_in_stream            ┆ 599048.526525 │
│ mc_share                  ┆ 191528.700917 │
│ dist_km                   ┆ 115268.281952 │
│ is_local                  ┆ 114893.148891 │
│ item_microcat_id          ┆ 84554.800184  │
│ gap_title                 ┆ 18987.038472  │
│ prox_title                ┆ 13273.175934  │
│ prox_filters              ┆ 12197.834712  │
│ gap_filters               ┆ 8849.710333   │
│ log_reviews               ┆ 7917.987312   │
│ log_price                 ┆ 4409.910282   │
│ item_rating               ┆ 3851.311534   │
│ had_local                 ┆ 1144.865206   │
│ item_is_phone_hidden      ┆ 272.91226     │
│ item_is_message_forbidde

In [25]:
import gc
for name in ["feat_rank", "pool_rank", "tr_df", "es_df", "X_tr", "X_es", "d_tr", "d_es",
             "pool_val", "feat_val", "scored", "top50", "X_val"]:
    globals().pop(name, None)
gc.collect()

172

In [33]:
N_CAND = 250
X_CAND = 50
bench = (pl.read_parquet(QUERIES_PATH)
         .with_columns(pl.col("search_query", "search_infm_params_text").fill_null(""))
         .with_row_index("qid"))

cols = {"qid": [], "item_id": [], "prox_title": [], "prox_filters": [], "is_local": []}
for q in tqdm(bench.iter_rows(named=True), total=bench.height):
    allowed = hard_filter(q["search_category"], q["search_infm_params_text"])
    _LOCAL, _NON_LOCAL = split_by_location(q["search_location_id"], allowed)
    res = search(q["search_query"], q["search_infm_params_text"], _LOCAL, _NON_LOCAL,
                 N_CAND, q["search_location_id"], X_CAND)
    for item_id, (pt, pf, il) in res.items():
        cols["qid"].append(q["qid"])
        cols["item_id"].append(item_id)
        cols["prox_title"].append(pt)
        cols["prox_filters"].append(pf)
        cols["is_local"].append(il)

pool_bench = pl.DataFrame(cols)
pool_bench.write_parquet(f"{CACHE_DIR}/pool_bench.parquet")
print(pool_bench.shape, pool_bench["qid"].n_unique())

100%|██████████| 2452/2452 [08:59<00:00,  4.54it/s]


(1101724, 5) 2230


In [34]:
qinfo = (bench.select("qid", "search_location_id")
         .join(centers, on="search_location_id", how="left")
         .with_columns(pl.col("c_lat").is_not_null().cast(pl.Int8).alias("had_local"))
         .drop("search_location_id"))

hist_all = (pl.read_parquet(TRAIN_PATH, columns=["search_query", "item_microcat_id"])
            .with_columns(pl.col("search_query").fill_null(""))
            .filter(pl.col("item_microcat_id").is_not_null()))
C_all = csr_matrix((np.ones(hist_all.height, dtype=np.float32),
                    ([t2i[t] for t in hist_all["search_query"].to_list()],
                     [mc2i[m] for m in hist_all["item_microcat_id"].to_list()])),
                   shape=(len(uniq_q), len(mc_vals)))
del hist_all

E_BQ = encode(bench["search_query"].to_list(), "query: ")
bqid = bench["qid"].to_numpy()

parts = []
for s in tqdm(range(0, len(E_BQ), 512)):
    S = E_BQ[s:s + 512] @ E_TR_Q.T
    top = np.argpartition(-S, K_SIM, axis=1)[:, :K_SIM]
    agg = np.vstack([np.asarray(C_all[t].sum(axis=0)).ravel() for t in top])
    tot = agg.sum(axis=1, keepdims=True)
    share = np.divide(agg, tot, out=np.zeros_like(agg), where=tot > 0)
    qi, mi = np.nonzero(share)
    parts.append(pl.DataFrame({"qid": bqid[s:s + 512][qi],
                               "item_microcat_id": mc_vals[mi],
                               "mc_share": share[qi, mi]}))
mc_share = pl.concat(parts)

feat_bench = build_features(pool_bench)
print(feat_bench.shape)
with pl.Config(tbl_rows=-1):
    print(feat_bench.null_count().transpose(include_header=True, column_names=["nulls"]))

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:04<00:00,  1.18it/s]


(1101724, 17)
shape: (17, 2)
┌───────────────────────────┬────────┐
│ column                    ┆ nulls  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ qid                       ┆ 0      │
│ item_id                   ┆ 0      │
│ prox_title                ┆ 0      │
│ prox_filters              ┆ 0      │
│ is_local                  ┆ 0      │
│ item_rating               ┆ 98552  │
│ item_microcat_id          ┆ 0      │
│ item_is_phone_hidden      ┆ 0      │
│ item_is_message_forbidden ┆ 0      │
│ had_local                 ┆ 0      │
│ mc_share                  ┆ 0      │
│ dist_km                   ┆ 181414 │
│ log_price                 ┆ 0      │
│ log_reviews               ┆ 71071  │
│ rank_in_stream            ┆ 0      │
│ gap_title                 ┆ 0      │
│ gap_filters               ┆ 0      │
└───────────────────────────┴────────┘


In [35]:
X_b = feat_bench.select(FEATS).to_pandas()
X_b[CAT_FEATS] = X_b[CAT_FEATS].astype("category")
scored_b = feat_bench.with_columns(
    pl.Series("score", ranker.predict(X_b, num_iteration=ranker.best_iteration)))

top50_b = (scored_b.sort(["qid", "score"], descending=[False, True])
           .group_by("qid", maintain_order=True).head(50)
           .group_by("qid", maintain_order=True).agg(pl.col("item_id")))

answer = (bench.select("qid", "query_id")
          .join(top50_b, on="qid", how="left")
          .with_columns(pl.col("item_id").fill_null([]).list.join(" ").alias("answer"))
          .select("query_id", "answer"))
answer.write_csv("/content/answer.csv")
answer.write_csv(f"{CACHE_DIR}/answer_v1.csv")
print(answer.head())

shape: (5, 2)
┌──────────────────┬─────────────────────────────────┐
│ query_id         ┆ answer                          │
│ ---              ┆ ---                             │
│ str              ┆ str                             │
╞══════════════════╪═════════════════════════════════╡
│ 70DfDUpwjxB4lzFd ┆ fa01652ca5f57d0f d722bcda1a555… │
│ JTrdTaZJvSiLPkXj ┆ 422d3ffdd5bbf626 cbeccbecb1fb8… │
│ LZCZNoVG4AFUkVRJ ┆ dab52187b4500d9b edecb3695ab88… │
│ 660ac9QVtXkRxZC3 ┆ 3cce43c811459438 4c15e7abba063… │
│ YgHcM9MVbxKnxD1e ┆ ba78ad593ec3412d 13f58fe9542bd… │
└──────────────────┴─────────────────────────────────┘


In [36]:
chk = pl.read_csv("/content/answer.csv", schema_overrides={"query_id": pl.Utf8, "answer": pl.Utf8})
corpus_ids = set(pl.read_parquet(ADS_PATH, columns=["item_id"])["item_id"].to_list())
bench_ids = bench["query_id"].to_list()

assert chk.columns == ["query_id", "answer"]
assert chk.height == len(bench_ids) and chk["query_id"].n_unique() == chk.height
assert set(chk["query_id"]) == set(bench_ids)
lens = []
for a in chk["answer"].fill_null(""):
    ids = a.split()
    lens.append(len(ids))
    assert len(ids) <= 50 and len(ids) == len(set(ids))
    assert all(re.fullmatch(r"[0-9a-f]{16}", x) and x in corpus_ids for x in ids)
print("OK | кандидатов на запрос: min", min(lens), "mean", np.mean(lens))

OK | кандидатов на запрос: min 0 mean 45.47308319738988


In [37]:
n_per_q = pool_bench.group_by("qid").len()
short = bench.join(n_per_q, on="qid", how="left").filter(pl.col("len").fill_null(0) < 50)
print(short.height)
print(short.select("search_query", "search_category", "search_infm_params_text", "len").head(20))
print(bench["search_category"].value_counts())

222
shape: (20, 4)
┌─────────────────────────────────┬─────────────────┬─────────────────────────┬──────┐
│ search_query                    ┆ search_category ┆ search_infm_params_text ┆ len  │
│ ---                             ┆ ---             ┆ ---                     ┆ ---  │
│ str                             ┆ i64             ┆ str                     ┆ u32  │
╞═════════════════════════════════╪═════════════════╪═════════════════════════╪══════╡
│ автомобиль бу под выкуп под та… ┆ 0               ┆                         ┆ null │
│ установка реечного потолка в в… ┆ 0               ┆                         ┆ null │
│ ищу спарринг партнера по тенни… ┆ 0               ┆                         ┆ null │
│ экскурсия в чечню               ┆ 0               ┆                         ┆ null │
│ репетитор по веб разработке     ┆ 0               ┆                         ┆ null │
│ …                               ┆ …               ┆ …                       ┆ …    │
│ роддом                